In [3]:
import os
import pandas as pd

# Path setup (notebook is inside python/notebooks/)
raw_dir = "../../data/raw"

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dfs = {}

print("=== 1. SHAPES, DUPLICATES & NULLS ===\n")
for name, filename in files.items():
    path = os.path.join(raw_dir, filename)
    df = pd.read_csv(path)
    dfs[name] = df
    
    dups = df.duplicated().sum()
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    
    print(f"--- {name.upper()} ---")
    print(f"Shape: {df.shape} | Exact Duplicates: {dups}")
    if len(nulls) > 0:
        print("Null columns:")
        for col, cnt in nulls.items():
            pct = (cnt / len(df)) * 100
            print(f"  - {col}: {cnt} ({pct:.2f}%)")
    else:
        print("Null columns: None")
    print()

print("=== 2. KEY TRAP CHECK: CUSTOMERS ===\n")
cust = dfs["customers"]
print(f"Total rows in customers: {len(cust)}")
print(f"Unique customer_id: {cust['customer_id'].nunique()}")
print(f"Unique customer_unique_id: {cust['customer_unique_id'].nunique()}")
print(f"Repeat customers (unique_id appearing > 1 time): {(cust['customer_unique_id'].value_counts() > 1).sum()}\n")

print("=== 3. ORDER STATUS BREAKDOWN ===\n")
orders = dfs["orders"]
status_counts = orders["order_status"].value_counts()
for status, cnt in status_counts.items():
    pct = (cnt / len(orders)) * 100
    print(f"{status}: {cnt} ({pct:.2f}%)")

=== 1. SHAPES, DUPLICATES & NULLS ===

--- CUSTOMERS ---
Shape: (99441, 5) | Exact Duplicates: 0
Null columns: None

--- GEOLOCATION ---
Shape: (1000163, 5) | Exact Duplicates: 261831
Null columns: None

--- ORDER_ITEMS ---
Shape: (112650, 7) | Exact Duplicates: 0
Null columns: None

--- ORDER_PAYMENTS ---
Shape: (103886, 5) | Exact Duplicates: 0
Null columns: None

--- ORDER_REVIEWS ---
Shape: (99224, 7) | Exact Duplicates: 0
Null columns:
  - review_comment_title: 87656 (88.34%)
  - review_comment_message: 58247 (58.70%)

--- ORDERS ---
Shape: (99441, 8) | Exact Duplicates: 0
Null columns:
  - order_approved_at: 160 (0.16%)
  - order_delivered_carrier_date: 1783 (1.79%)
  - order_delivered_customer_date: 2965 (2.98%)

--- PRODUCTS ---
Shape: (32951, 9) | Exact Duplicates: 0
Null columns:
  - product_category_name: 610 (1.85%)
  - product_name_lenght: 610 (1.85%)
  - product_description_lenght: 610 (1.85%)
  - product_photos_qty: 610 (1.85%)
  - product_weight_g: 2 (0.01%)
  - product

In [5]:
for name in ["orders", "products", "sellers", "category_translation"]:
    df = dfs[name]
    dups = df.duplicated().sum()
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"--- {name.upper()} ---")
    print(f"Shape: {df.shape} | Exact Duplicates: {dups}")
    for col, cnt in nulls.items():
        print(f"  - {col}: {cnt} ({cnt/len(df)*100:.2f}%)")
    if len(nulls) == 0:
        print("  Nulls: None")

cust = dfs["customers"]
print("\n--- CUSTOMERS KEY CHECK ---")
print(f"Total rows: {len(cust)} | customer_id: {cust['customer_id'].nunique()} | customer_unique_id: {cust['customer_unique_id'].nunique()} | Repeat customers: {(cust['customer_unique_id'].value_counts() > 1).sum()}")

print("\n--- FULL ORDER STATUS ---")
print(dfs["orders"]["order_status"].value_counts().to_dict())

--- ORDERS ---
Shape: (99441, 8) | Exact Duplicates: 0
  - order_approved_at: 160 (0.16%)
  - order_delivered_carrier_date: 1783 (1.79%)
  - order_delivered_customer_date: 2965 (2.98%)
--- PRODUCTS ---
Shape: (32951, 9) | Exact Duplicates: 0
  - product_category_name: 610 (1.85%)
  - product_name_lenght: 610 (1.85%)
  - product_description_lenght: 610 (1.85%)
  - product_photos_qty: 610 (1.85%)
  - product_weight_g: 2 (0.01%)
  - product_length_cm: 2 (0.01%)
  - product_height_cm: 2 (0.01%)
  - product_width_cm: 2 (0.01%)
--- SELLERS ---
Shape: (3095, 4) | Exact Duplicates: 0
  Nulls: None
--- CATEGORY_TRANSLATION ---
Shape: (71, 2) | Exact Duplicates: 0
  Nulls: None

--- CUSTOMERS KEY CHECK ---
Total rows: 99441 | customer_id: 99441 | customer_unique_id: 96096 | Repeat customers: 2997

--- FULL ORDER STATUS ---
{'delivered': 96478, 'shipped': 1107, 'canceled': 625, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'created': 5, 'approved': 2}


In [6]:
rev = dfs["order_reviews"]
dup_rev_ids = rev["review_id"].duplicated().sum()
print(f"1. Duplicate review_id count: {dup_rev_ids}")

# 2. Check missing category translations
prod_cats = set(dfs["products"]["product_category_name"].dropna().unique())
trans_cats = set(dfs["category_translation"]["product_category_name"].unique())
missing_translations = prod_cats - trans_cats
print(f"2. Categories in products missing from translation table: {missing_translations}")

# 3. Referential Integrity (FK) Checks
orders_set = set(dfs["orders"]["order_id"])
cust_set = set(dfs["customers"]["customer_id"])
prod_set = set(dfs["products"]["product_id"])
seller_set = set(dfs["sellers"]["seller_id"])

print(f"3. Orphan orders (customer_id not in customers): {len(set(dfs['orders']['customer_id']) - cust_set)}")
print(f"4. Orphan order_items (order_id not in orders): {len(set(dfs['order_items']['order_id']) - orders_set)}")
print(f"5. Orphan order_items (product_id not in products): {len(set(dfs['order_items']['product_id']) - prod_set)}")
print(f"6. Orphan order_items (seller_id not in sellers): {len(set(dfs['order_items']['seller_id']) - seller_set)}")
print(f"7. Orphan order_payments (order_id not in orders): {len(set(dfs['order_payments']['order_id']) - orders_set)}")
print(f"8. Orphan order_reviews (order_id not in orders): {len(set(dfs['order_reviews']['order_id']) - orders_set)}")

1. Duplicate review_id count: 814
2. Categories in products missing from translation table: {'portateis_cozinha_e_preparadores_de_alimentos', 'pc_gamer'}
3. Orphan orders (customer_id not in customers): 0
4. Orphan order_items (order_id not in orders): 0
5. Orphan order_items (product_id not in products): 0
6. Orphan order_items (seller_id not in sellers): 0
7. Orphan order_payments (order_id not in orders): 0
8. Orphan order_reviews (order_id not in orders): 0


In [7]:
items = dfs["order_items"]
print(f"9. Items with price <= 0: {(items['price'] <= 0).sum()} | freight_value < 0: {(items['freight_value'] < 0).sum()}")


9. Items with price <= 0: 0 | freight_value < 0: 0


In [8]:
import unicodedata

cleaned_dir = "../../data/cleaned"
os.makedirs(cleaned_dir, exist_ok=True)

def normalize_text(text):
    if pd.isna(text):
        return text
    # Remove Portuguese accents (e.g., são paulo -> sao paulo) & lowercase/strip
    nfkd = unicodedata.normalize('NFKD', str(text))
    ascii_str = "".join([c for c in nfkd if not unicodedata.combining(c)])
    return ascii_str.strip().lower()

# 1. CUSTOMERS
customers_clean = dfs["customers"].copy()
customers_clean["customer_city"] = customers_clean["customer_city"].apply(normalize_text)
customers_clean["customer_state"] = customers_clean["customer_state"].str.strip().str.upper()

# 2. SELLERS
sellers_clean = dfs["sellers"].copy()
sellers_clean["seller_city"] = sellers_clean["seller_city"].apply(normalize_text)
sellers_clean["seller_state"] = sellers_clean["seller_state"].str.strip().str.upper()

# 3. GEOLOCATION (Deduplicate & aggregate by zip_code_prefix)
geo_temp = dfs["geolocation"].copy()
geo_temp["geolocation_city"] = geo_temp["geolocation_city"].apply(normalize_text)
geo_temp["geolocation_state"] = geo_temp["geolocation_state"].str.strip().str.upper()

geolocation_clean = (
    geo_temp.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0]),
        geolocation_state=("geolocation_state", lambda x: x.mode()[0] if not x.mode().empty else x.iloc[0])
    )
)

# 4. CATEGORY TRANSLATION (Add missing 2 categories + uncategorized)
cat_trans_clean = dfs["category_translation"].copy()
extra_cats = pd.DataFrame([
    {"product_category_name": "portateis_cozinha_e_preparadores_de_alimentos", "product_category_name_english": "portable_kitchen_and_food_preparers"},
    {"product_category_name": "pc_gamer", "product_category_name_english": "pc_gamer"},
    {"product_category_name": "uncategorized", "product_category_name_english": "uncategorized"}
])
cat_trans_clean = pd.concat([cat_trans_clean, extra_cats], ignore_index=True).drop_duplicates(subset=["product_category_name"])

# 5. PRODUCTS (Fix typos, fill missing category & median dimensions)
products_clean = dfs["products"].copy()
products_clean = products_clean.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})
products_clean["product_category_name"] = products_clean["product_category_name"].fillna("uncategorized")
for col in ["product_name_length", "product_description_length", "product_photos_qty"]:
    products_clean[col] = products_clean[col].fillna(0).astype(int)

for col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products_clean[col] = products_clean[col].fillna(products_clean[col].median())

# 6. ORDERS (Parse datetimes & engineer delivery KPIs)
orders_clean = dfs["orders"].copy()
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in date_cols:
    orders_clean[col] = pd.to_datetime(orders_clean[col])

# Delivery metrics (calculated where delivered_customer_date is available)
orders_clean["actual_delivery_days"] = (
    (orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"]).dt.total_seconds() / 86400
).round(2)
orders_clean["estimated_delivery_days"] = (
    (orders_clean["order_estimated_delivery_date"] - orders_clean["order_purchase_timestamp"]).dt.total_seconds() / 86400
).round(2)
# Positive = late, Negative = early
orders_clean["delivery_delay_days"] = (
    (orders_clean["order_delivered_customer_date"] - orders_clean["order_estimated_delivery_date"]).dt.total_seconds() / 86400
).round(2)
orders_clean["is_late"] = (orders_clean["delivery_delay_days"] > 0).astype(int)
orders_clean.loc[orders_clean["order_delivered_customer_date"].isna(), "is_late"] = pd.NA

# 7. ORDER ITEMS
order_items_clean = dfs["order_items"].copy()
order_items_clean["shipping_limit_date"] = pd.to_datetime(order_items_clean["shipping_limit_date"])
order_items_clean["total_item_value"] = (order_items_clean["price"] + order_items_clean["freight_value"]).round(2)

# 8. ORDER PAYMENTS
order_payments_clean = dfs["order_payments"].copy()

# 9. ORDER REVIEWS (Clean newlines in text so CSV export never breaks rows)
order_reviews_clean = dfs["order_reviews"].copy()
order_reviews_clean["review_creation_date"] = pd.to_datetime(order_reviews_clean["review_creation_date"])
order_reviews_clean["review_answer_timestamp"] = pd.to_datetime(order_reviews_clean["review_answer_timestamp"])
for col in ["review_comment_title", "review_comment_message"]:
    order_reviews_clean[col] = order_reviews_clean[col].str.replace(r"[\r\n]+", " ", regex=True).str.strip()

# Save all 9 cleaned CSVs
cleaned_tables = {
    "customers_cleaned.csv": customers_clean,
    "sellers_cleaned.csv": sellers_clean,
    "geolocation_cleaned.csv": geolocation_clean,
    "category_translation_cleaned.csv": cat_trans_clean,
    "products_cleaned.csv": products_clean,
    "orders_cleaned.csv": orders_clean,
    "order_items_cleaned.csv": order_items_clean,
    "order_payments_cleaned.csv": order_payments_clean,
    "order_reviews_cleaned.csv": order_reviews_clean
}

print("=== CLEANED TABLES SUMMARY ===")
for fname, df_clean in cleaned_tables.items():
    out_path = os.path.join(cleaned_dir, fname)
    df_clean.to_csv(out_path, index=False)
    print(f"Saved {fname:35s} | Shape: {df_clean.shape}")

=== CLEANED TABLES SUMMARY ===
Saved customers_cleaned.csv               | Shape: (99441, 5)
Saved sellers_cleaned.csv                 | Shape: (3095, 4)
Saved geolocation_cleaned.csv             | Shape: (19015, 5)
Saved category_translation_cleaned.csv    | Shape: (74, 2)
Saved products_cleaned.csv                | Shape: (32951, 9)
Saved orders_cleaned.csv                  | Shape: (99441, 12)
Saved order_items_cleaned.csv             | Shape: (112650, 8)
Saved order_payments_cleaned.csv          | Shape: (103886, 5)
Saved order_reviews_cleaned.csv           | Shape: (99224, 7)
